In [1]:
import duckdb

In [2]:
# Q1: Total number of event rows
query_1 = """
SELECT COUNT(*) 
FROM '../data/processed_nov/*.parquet'
"""
print("Total Rows:", duckdb.query(query_1).fetchall())

Total Rows: [(67501979,)]


In [3]:
# Q2: How many unique user_session IDs exist?
query_2 = """
SELECT COUNT(DISTINCT user_session) 
FROM '../data/processed_nov/*.parquet'
"""
print("Unique Sessions:", duckdb.query(query_2).fetchall())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique Sessions: [(13776050,)]


In [4]:
# Q3: Total count of each event_type
query_3 = """
SELECT event_type, COUNT(*) as count 
FROM '../data/processed_nov/*.parquet' 
GROUP BY event_type
"""
print("Event Type Counts:\n", duckdb.query(query_3).df()) # .df() returns a nice Pandas dataframe!

Event Type Counts:
   event_type     count
0       view  63556110
1       cart   3028930
2   purchase    916939


In [5]:
# Q4: Number of sessions with more than 1 event
query_4 = """
SELECT user_session, COUNT(*) as event_count 
FROM '../data/processed_nov/*.parquet' 
GROUP BY user_session HAVING COUNT(*) > 1
"""
print("Sessions with More than 1 Event:\n", duckdb.query(query_4).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sessions with More than 1 Event:
                                  user_session  event_count
0        4d3b30da-a5e4-49df-b1a8-ba5943f1dd33            9
1        755422e7-9040-477b-9bd2-6a6e8fd97387           11
2        313628f1-68b8-460d-84f6-cec7a8796ef2            3
3        5e6ef132-4d7c-4730-8c7f-85aa4082588f            9
4        0a899268-31eb-46de-898d-09b2da950b24            3
...                                       ...          ...
8775079  4bea3ad0-22c7-41ac-853b-229e6267c487            3
8775080  5c8456d8-4670-4029-b0ba-6438113ddf98            2
8775081  dbca9f12-9351-48ac-98a6-cc650102b47e            2
8775082  720e1344-3a02-4671-be75-a7ac7ae3fa0b            2
8775083  a87417b2-5f76-4be8-aede-8b6f2c633ed7            2

[8775084 rows x 2 columns]


In [8]:
# Q5: Ordered sequence of items per session
query_5_safe = """
WITH raw_subset AS (
    SELECT user_session, product_id, event_time
    FROM '../data/processed_nov/*.parquet'
    LIMIT 100000
)
SELECT 
    user_session, 
    list(product_id ORDER BY event_time ASC) as ordered_product_ids,
    COUNT(*) as sequence_length,
    ordered_product_ids[1:-2] as input_sequence,
    ordered_product_ids[-1] as target_item
FROM raw_subset 
GROUP BY user_session 
HAVING COUNT(*) > 1 
LIMIT 5;
"""
print("Ordered Product IDs per Session:\n", duckdb.query(query_5_safe).df())

Ordered Product IDs per Session:
                            user_session  \
0  4d3b30da-a5e4-49df-b1a8-ba5943f1dd33   
1  755422e7-9040-477b-9bd2-6a6e8fd97387   
2  313628f1-68b8-460d-84f6-cec7a8796ef2   
3  5e6ef132-4d7c-4730-8c7f-85aa4082588f   
4  0a899268-31eb-46de-898d-09b2da950b24   

                                 ordered_product_ids  sequence_length  \
0  [1003461, 1004184, 1003461, 1005234, 1003898, ...                9   
1  [17302664, 17301495, 17301611, 17301421, 17301...               11   
2                        [1004775, 1004775, 1004775]                3   
3  [15900065, 15900065, 15900117, 15900952, 15900...                9   
4                     [12708937, 12708937, 12703015]                3   

                                      input_sequence  target_item  
0  [1003461, 1004184, 1003461, 1005234, 1003898, ...      1003898  
1  [17302664, 17301495, 17301611, 17301421, 17301...     17301725  
2                                 [1004775, 1004775]      100477